# VMTF-Net — Chạy trên Google Colab Extension / Local

**Lưu ý quan trọng:**
- Nếu chọn Kernel **Colab (T4/V100/A100)**: code chạy trên máy ảo Google Colab (`/content/...`), **không tự thấy ổ `D:\` trên máy Windows**.
- Để Colab dùng dữ liệu thật từ `D:\PROJECT\vietnamese-multimodal-fake-news-detection\data`, chọn 1 trong 2 cách:
  1. Đặt `DATA_SOURCE = 'upload'` ở Cell 3 để chọn file `data.zip` (hoặc `vmtf_candidates.csv`) tải lên Colab.
  2. Đặt `DATA_SOURCE = 'drive'` và chép thư mục `data/` lên Google Drive tại `MyDrive/vmtf/data/`.
- Nếu chỉ muốn test nhanh code/GPU, giữ `DATA_SOURCE = 'auto'` (tự dùng `data/` nếu có, chưa có thì tạo 4 mẫu giả lập).

In [43]:
# 1. Kiểm tra GPU trên Runtime
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv || echo "Đang chạy CPU runtime (hãy đổi Kernel sang GPU nếu muốn train nhanh)"

name, memory.total [MiB], driver_version
Tesla T4, 15360 MiB, 580.82.07


In [44]:
# 2. Thiết lập thư mục làm việc (Colab VM hoặc Local) và cài thư viện bổ sung
import os
IN_COLAB = os.path.exists('/content')
if IN_COLAB:
    %cd /content
    if not os.path.exists('/content/vietnamese-multimodal-fake-news-detection'):
        !git clone -q https://github.com/kascik04/vietnamese-multimodal-fake-news-detection.git
    else:
        !git -C /content/vietnamese-multimodal-fake-news-detection pull -q
    %cd /content/vietnamese-multimodal-fake-news-detection
    !pip install -q pyvi imagehash
elif os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

print('Working dir:', os.getcwd(), '| IN_COLAB:', IN_COLAB)
!git log --oneline -1

/content
Updating files: 100% (2915/2915), done.
/content/vietnamese-multimodal-fake-news-detection
Working dir: /content/vietnamese-multimodal-fake-news-detection | IN_COLAB: True
71c4444 (HEAD -> main, origin/main, origin/HEAD) up


In [45]:
# 3. Nạp dữ liệu thật (Local / Drive / Upload) hoặc tạo 4 mẫu smoke-test
import os, sys, shutil, zipfile, subprocess
import pandas as pd
from PIL import Image, ImageDraw

DATA_SOURCE = 'auto'   # 'auto' | 'drive' | 'upload' | 'smoke_test'
DRIVE_DATA_DIR = '/content/drive/MyDrive/vmtf/data'

os.makedirs('data/images', exist_ok=True)

if IN_COLAB and DATA_SOURCE == 'drive':
    from google.colab import drive
    drive.mount('/content/drive')
    if os.path.exists(DRIVE_DATA_DIR):
        shutil.copytree(DRIVE_DATA_DIR, 'data', dirs_exist_ok=True)
        print('✅ Đã copy dữ liệu từ Google Drive:', DRIVE_DATA_DIR)
    else:
        raise FileNotFoundError(f'Không tìm thấy {DRIVE_DATA_DIR} trên Google Drive!')

elif IN_COLAB and DATA_SOURCE == 'upload':
    from google.colab import files
    uploaded = files.upload()
    for fname in uploaded:
        if fname.endswith('.zip'):
            with zipfile.ZipFile(fname, 'r') as zf:
                zf.extractall('data')
        elif fname.endswith('.csv'):
            shutil.move(fname, os.path.join('data', os.path.basename(fname)))
    print('✅ Đã nhận file upload vào data/')

IS_SMOKE_TEST = False
# Nếu data/train.csv chỉ là 4 mẫu smoke-test cũ nhưng đã có vmtf_candidates.csv -> xoá 4 mẫu cũ để nạp dữ liệu thật
if os.path.exists('data/train.csv') and os.path.exists('data/vmtf_candidates.csv'):
    try:
        if len(pd.read_csv('data/train.csv')) <= 4:
            for _f in ('data/train.csv', 'data/val.csv', 'data/test.csv'):
                if os.path.exists(_f):
                    os.remove(_f)
    except Exception:
        pass

if DATA_SOURCE != 'smoke_test' and not os.path.exists('data/train.csv') and os.path.exists('data/vmtf_candidates.csv'):
    df_raw = pd.read_csv('data/vmtf_candidates.csv')
    labeled = df_raw[df_raw['label'].isin([0, 1, '0', '1', 0.0, 1.0])]
    if len(labeled) < 6:
        print('⚙️ Phát hiện data/vmtf_candidates.csv chưa gán nhãn -> đang tự động gán nhãn sơ bộ (weak supervision)...')
        import re
        TRUSTED = ['vtv', 'chuyển động 24h', 'ấn tượng vtv', 'vtc', 'antv', 'công an nhân dân', 'nhân dân', 'quốc hội', 'qpvn', 'vov', 'thông tấn xã', 'ttxvn', 'hanoitv', 'htv', 'thvl', 'vinh long', 'đồng tháp', 'thđt', 'nghệ an tv', 'bptv', 'anninhthudo', 'an ninh thủ đô', 'đài phát thanh', 'truyền hình', 'sức khỏe & đời sống', 'sức khỏe và đời sống', 'thanh niên', 'tuổi trẻ', 'tiền phong', 'viettimes', 'lao động', 'dân trí', 'vnexpress', 'vietnamnet', 'pháp luật', 'người lao động', 'chính phủ', 'alobacsi', 'bệnh viện', 'bộ y tế', 'vnvc', 'tâm anh', 'medlatec', 'vinmec', 'đại học y']
        FAKE_SRC = ['đi soi sao', 'đông dược', 'gia truyền', 'lương y', 'thần y', 'thực dưỡng', 'chân thiện mỹ', 'năng lượng tái sinh', 'năng lượng gốc', 'chữa lành', 'người nổi tiếng', 'chuyện nghệ sĩ', 'xóm hóng', '69smedia', 'đầu xanh mỏ hỗn', 'nghiện hay chuyện', 'hóng biến', 'sao việt', 'bí mật', 'tâm linh', 'phong thủy', 'tử vi', 'thầy bói', 'cây thuốc', 'bài thuốc']
        REAL_TXT = ['cảnh báo lừa đảo', 'công an cảnh báo', 'bác bỏ tin đồn', 'thông tin sai sự thật', 'xử phạt', 'khởi tố', 'bắt tạm giam', 'triệt phá', 'cục an toàn thực phẩm', 'bộ y tế', 'bộ công an', 'đính chính', 'cảnh giác', 'cơ quan chức năng', 'bản tin', 'thời sự', 'khuyến cáo', 'cảnh báo thủ đoạn']
        FAKE_TXT = ['trị dứt điểm', 'chữa dứt điểm', 'chữa bách bệnh', 'trị bách bệnh', 'chữa khỏi hoàn toàn', 'bài thuốc dân gian', 'thần dược', 'nước kiềm', 'thực dưỡng', 'không cần thuốc', 'gây đột quỵ', 'bằng tâm linh', 'trục vong', 'bắt ma', 'trừ tà', 'bùa ngải', 'ngày tận thế', 'cúng giải hạn', 'lộ clip nóng', 'qua đời đột ngột', 'bóc phốt', 'phong sát', 'ngoại tình', 'vỡ nợ tiền tỷ', 'mổ nội tạng', 'việc nhẹ lương cao', 'cam kết lãi', 'sàn giao dịch ảo', 'app kiếm tiền', 'chấn động', 'kinh hoàng', 'rúng động', 'tại nhà', 'dân gian', 'mẹo chữa', 'khỏi hẳn', 'xôn xao', 'thực hư']
        RUMOR_Q = {'drama showbiz việt', 'sao kê từ thiện nghệ sĩ', 'nghệ sĩ qua đời đột ngột', 'bóc phốt nghệ sĩ', 'thuốc gia truyền trị dứt điểm', 'lộ clip nóng showbiz', 'thực hư nghệ sĩ qua đời', 'bắt cóc trẻ em mổ nội tạng', 'chữa bệnh qua mạng xã hội', 'scandal ca sĩ diễn viên', 'thần y trị bách bệnh', 'bài thuốc dân gian chữa ung thư', 'uống nước kiềm chữa ung thư', 'chữa bệnh bằng tâm linh', 'phong sát nghệ sĩ', 'tin đồn nghệ sĩ', 'thực dưỡng chữa bách bệnh', 'vắc xin gây đột quỵ', 'thực phẩm chức năng trị bách bệnh', 'nghệ sĩ vỡ nợ tiền tỷ', 'tin đồn ly hôn ngoại tình showbiz', 'đa cấp biến tướng', 'tín dụng đen bốc bát họ', 'đầu tư tiền ảo cam kết lãi', 'phương thuốc', 'thuốc chữa bệnh', 'thực phẩm chức năng', 'sự thật vụ án chấn động', 'tin đồn vỡ đập', 'tin đồn thất thiệt hoang mang'}
        REAL_Q = {'mưa lớn', 'ngập lụt', 'lũ quét', 'lũ lụt', 'sạt lở', 'bão', 'hạn hán', 'động đất', 'thiên tai', 'giao thông', 'sức khỏe', 'bệnh viện', 'y tế', 'điều trị', 'cúm', 'covid', 'vaccine', 'vắc xin', 'dịch bệnh', 'xử phạt tung tin giả', 'ngộ độc thực phẩm chức năng giả'}
        def _auto_lbl(row):
            src = str(row.get('source', '') or '').lower()
            t_raw = str(row.get('text', '') or '')
            t = t_raw.lower()
            n = str(row.get('notes', '') or '').lower()
            m = re.search(r'query=([^;]+)', n)
            q = m.group(1).strip() if m else ''
            s = 0
            if any(k in src for k in TRUSTED): s -= 3
            if any(k in src for k in FAKE_SRC): s += 3
            s -= sum(1 for k in REAL_TXT if k in t)
            s += sum(1 for k in FAKE_TXT if k in t)
            if re.search(r'\b0[35789]\d{8}\b', t): s += 2
            if '!' in t_raw or '?' in t_raw: s += 1
            if q in RUMOR_Q: s += 1
            if q in REAL_Q: s -= 1
            return 1 if s > 0 else 0
        df_raw['label'] = df_raw.apply(_auto_lbl, axis=1)
        df_raw['raw_label'] = df_raw['label'].map({0: 'real', 1: 'fake'})
        df_raw['label_source'] = 'weak_supervision_v1'
        df_raw['annotation_status'] = 'auto_labeled'
        df_raw.to_csv('data/vmtf_candidates.csv', index=False)
        labeled = df_raw
    print(f'✅ Tìm thấy {len(labeled)}/{len(df_raw)} mẫu đã gán nhãn -> đang chia train/val/test...')
    subprocess.run([
        sys.executable, 'scripts/validate_and_preprocess_dataset.py',
        '--input_csv', 'data/vmtf_candidates.csv',
        '--image_dir', 'data/images',
        '--output_dir', 'data',
    ], check=True)

if DATA_SOURCE == 'smoke_test' or not os.path.exists('data/train.csv'):
    IS_SMOKE_TEST = True
    print('⚠️ Không thấy dữ liệu thật trên runtime -> tạo 4 mẫu giả lập để smoke-test pipeline.')
    with open('data/train.csv', 'w', encoding='utf-8') as f:
        f.write(
            'sample_id,text,image_1_local_path,like_count,comment_count,share_count,angry_count,haha_count,label\n'
            'post_001,Bộ Y Tế chính thức bác bỏ tin đồn thất thiệt lan truyền trên mạng xã hội sáng nay.,news_1.jpg,1250,340,120,15,30,0\n'
            'post_002,Cảnh báo khẩn cấp: Phương pháp chữa bách bệnh bằng lá cây chưa qua kiểm chứng khoa học!,news_2.jpg,89000,15400,45000,31150,35600,1\n'
            'post_003,Chính phủ phê duyệt dự án phát triển công nghệ cao và trí tuệ nhân tạo quốc gia.,news_3.jpg,4500,820,910,10,45,0\n'
            'post_004,Sốc: Hình ảnh sinh vật lạ xuất hiện tại vùng biển miền Trung gây hoang mang dư luận.,news_4.jpg,52000,8700,19500,13000,28600,1\n'
        )
    with open('data/val.csv', 'w', encoding='utf-8') as f:
        f.write(
            'sample_id,text,image_1_local_path,like_count,comment_count,share_count,angry_count,haha_count,label\n'
            'post_001,Bộ Y Tế chính thức bác bỏ tin đồn thất thiệt lan truyền trên mạng xã hội sáng nay.,news_1.jpg,1250,340,120,15,30,0\n'
            'post_002,Cảnh báo khẩn cấp: Phương pháp chữa bách bệnh bằng lá cây chưa qua kiểm chứng khoa học!,news_2.jpg,89000,15400,45000,31150,35600,1\n'
        )
    with open('data/test.csv', 'w', encoding='utf-8') as f:
        f.write(
            'sample_id,text,image_1_local_path,like_count,comment_count,share_count,angry_count,haha_count,label\n'
            'post_003,Chính phủ phê duyệt dự án phát triển công nghệ cao và trí tuệ nhân tạo quốc gia.,news_3.jpg,4500,820,910,10,45,0\n'
            'post_004,Sốc: Hình ảnh sinh vật lạ xuất hiện tại vùng biển miền Trung gây hoang mang dư luận.,news_4.jpg,52000,8700,19500,13000,28600,1\n'
        )
    colors = {'news_1.jpg': (30, 136, 229), 'news_2.jpg': (229, 57, 53), 'news_3.jpg': (67, 160, 71), 'news_4.jpg': (251, 140, 0)}
    for name, rgb in colors.items():
        p = os.path.join('data/images', name)
        if not os.path.exists(p):
            img = Image.new('RGB', (224, 224), rgb)
            ImageDraw.Draw(img).rectangle([40, 40, 184, 184], outline=(255, 255, 255), width=4)
            img.save(p)

for split in ('train', 'val', 'test'):
    p = f'data/{split}.csv'
    if os.path.exists(p):
        df_s = pd.read_csv(p)
        print(f'{split}.csv: {len(df_s)} mẫu | phân bố nhãn: {df_s["label"].value_counts().to_dict()}')

train.csv: 4 mẫu | phân bố nhãn: {0: 2, 1: 2}
val.csv: 2 mẫu | phân bố nhãn: {0: 1, 1: 1}
test.csv: 2 mẫu | phân bố nhãn: {0: 1, 1: 1}


In [ ]:
# 4. Chạy Offline Unit Tests (tùy chọn) & Train VMTF-Net
import os, pandas as pd

RUN_UNIT_TESTS = False
if RUN_UNIT_TESTS and os.path.exists('test_step1.py'):
    !python test_step1.py && python test_step2.py && python test_step3.py && python test_step4.py && python test_step5.py

n_train = len(pd.read_csv('data/train.csv'))
EXP_DIR = 'checkpoints/colab_ext_smoke_test' if IS_SMOKE_TEST else 'checkpoints/colab_ext_proposed_seed42'
BATCH   = 2 if n_train < 16 else 16
ACCUM   = 1 if n_train < 16 else 2   # effective batch = 32 mà không tràn VRAM T4
EPOCHS  = 2 if IS_SMOKE_TEST else 15
N_BOOT  = 50 if IS_SMOKE_TEST else 1000
OVERWRITE = False  # True nếu muốn train lại từ đầu, xóa checkpoint cũ
CACHE_DIR = '/content/cache' if IN_COLAB else 'data/cache'

last_ckpt = f'{EXP_DIR}/last_checkpoint.pt'
resume_flag = f'--resume {last_ckpt}' if (not OVERWRITE and os.path.exists(last_ckpt)) else '--overwrite'

!python -m vmtf_net.train \
    --train_csv data/train.csv \
    --val_csv data/val.csv \
    --test_csv data/test.csv \
    --image_dir data/images \
    --cache_dir {CACHE_DIR} \
    --fusion_type hybrid_proposed \
    --consistency_lambda 0.1 \
    --social_preset basic \
    --batch_size {BATCH} \
    --grad_accum_steps {ACCUM} \
    --epochs {EPOCHS} \
    --lr 2e-5 \
    --head_lr 1e-4 \
    --n_bootstrap {N_BOOT} \
    --checkpoint_dir {EXP_DIR} \
    --no_tqdm {resume_flag}

VMTFConfig (Vietnamese Multimodal Transformer Fusion Network)
  - Device:              cuda (AMP: True, Multi-GPU: True)
  - Text Backbone:       vinai/phobert-base (Max Seq: 256, Pool: cls, Seg: auto)
  - Visual Backbone:     google/vit-base-patch16-224 (Image: 224x224)
  - Social Features (3): ['share_count', 'like_count', 'comment_count']
  - Experiment:          hybrid_proposed_phobert-base_vit-base-patch16-224_social-basic_lambda0.1_seed42
  - Batch Size:          2 x accum 1 (Val: 32)
  - Learning Rates:      Backbone: 2e-05, Head: 0.0001
  - Loss Config:         Lambda Consistency: 0.1, Focal: False, Class weights: none
  - Num Workers:         7
2026-11-13 16:54:19,056 - INFO - Loading text tokenizer: vinai/phobert-base
2026-11-13 16:54:19,169 - INFO - HTTP Request: GET https://huggingface.co/api/models/vinai/phobert-base "HTTP/1.1 200 OK"
2026-11-13 16:54:19,219 - INFO - HTTP Request: GET https://huggingface.co/api/models/vinai/phobert-base/tree/main/additional_chat_templates?

In [42]:
# 5. Xem kết quả train & thông số GPU
import json
import pandas as pd

display(pd.read_csv(f'{EXP_DIR}/history.csv'))
print('--- run_info.json ---')
print(json.dumps(json.load(open(f'{EXP_DIR}/run_info.json')), indent=2, ensure_ascii=False))
print('--- test metrics ---')
print(json.dumps(json.load(open(f'{EXP_DIR}/results.json'))['test']['metrics'], indent=2, ensure_ascii=False))

,epoch,train_total_loss,train_cls_loss,train_consistency_loss,train_gate_entropy,train_accuracy,train_f1_macro,lr_backbone,val_threshold,val_n_samples,...,val_recall_macro,val_precision_fake,val_recall_fake,val_auc_roc,val_pr_auc,val_brier,val_ece,val_loss,val_avg_beta,epoch_seconds
0,1,0.773545,0.719088,0.544566,0.673453,0.25,0.200000,5.000000e-06,0.5,2,...,0.5,0.5,1.0,1.0,1.0,0.176298,0.408270,0.594167,0.607391,3.204822
1,2,0.855309,0.804693,0.506156,0.682336,0.25,0.200000,1.500000e-05,0.5,2,...,0.5,0.5,1.0,1.0,1.0,0.189830,0.389078,0.600200,0.623784,0.476887
2,3,0.636126,0.586885,0.492417,0.674273,0.75,0.733333,1.996195e-05,0.5,2,...,1.0,1.0,1.0,1.0,1.0,0.071288,0.248497,0.339466,0.631511,1.853180
3,4,0.291368,0.250814,0.405545,0.676612,1.00,1.000000,1.965926e-05,0.5,2,...,1.0,1.0,1.0,1.0,1.0,0.021046,0.134813,0.183185,0.623553,1.873054
4,5,0.147008,0.110199,0.368090,0.684096,1.00,1.000000,1.906308e-05,0.5,2,...,1.0,1.0,1.0,1.0,1.0,0.003260,0.056523,0.087978,0.587994,1.754937
5,6,0.114561,0.081615,0.329464,0.689439,1.00,1.000000,1.819152e-05,0.5,2,...,1.0,1.0,1.0,1.0,1.0,0.000899,0.029254,0.054263,0.524510,2.081918
6,7,0.058624,0.031058,0.275660,0.664587,1.00,1.000000,1.707107e-05,0.5,2,...,1.0,1.0,1.0,1.0,1.0,0.000209,0.014361,0.032319,0.591682,2.155018
7,8,0.052801,0.032096,0.207048,0.664670,1.00,1.000000,1.573576e-05,0.5,2,...,1.0,1.0,1.0,1.0,1.0,0.000079,0.008870,0.021053,0.632520,2.158397
8,9,0.028623,0.012195,0.164277,0.646704,1.00,1.000000,1.422618e-05,0.5,2,...,1.0,1.0,1.0,1.0,1.0,0.000043,0.006444,0.015273,0.627235,2.141875
9,10,0.019886,0.006917,0.129691,0.666115,1.00,1.000000,1.258819e-05,0.5,2,...,1.0,1.0,1.0,1.0,1.0,0.000028,0.005242,0.012064,0.614371,2.134562


--- run_info.json ---
{
  "experiment": "hybrid_proposed_phobert-base_vit-base-patch16-224_social-basic_lambda0.1_seed42",
  "environment": {
    "python": "3.13.16",
    "platform": "Linux-6.6.159+-x86_64-with-glibc2.39",
    "torch": "2.14.1+cu130",
    "transformers": "5.22.1",
    "cuda_available": true,
    "gpu_count": 1,
    "gpus": [
      "Tesla T4"
    ],
    "kaggle": true,
    "colab": true
  },
  "parameters": {
    "total": 228489220,
    "trainable": 228489220
  },
  "precision": {
    "amp": true,
    "amp_dtype": "float16",
    "tf32": false,
    "num_workers": 7,
    "gpus_used": 1
  },
  "train_samples": 4,
  "val_samples": 2,
  "test_samples": 2,
  "best_epoch": 20,
  "avg_epoch_seconds": 2.0361339826000404,
  "total_run_seconds": 204.90267340300034,
  "peak_gpu_memory_gb": 4.247576713562012,
  "test_samples_per_second": 4.345233078561598
}
--- test metrics ---
{
  "threshold": 0.05,
  "n_samples": 2,
  "accuracy": 1.0,
  "f1_macro": 1.0,
  "f1_fake": 1.0,
  "precis